# 02 · Đánh giá

1. Một lượt chạy trên train (có đáp án): điểm, theo loại / shop / dạng câu, nhóm lỗi, ngưỡng bỏ phiếu.
2. Các bản nộp public: so với nhau, không cần nhãn.

Output chứa dữ liệu thi: **xóa output trước khi commit**.

In [ ]:
import json, re, sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"
read = lambda p: [json.loads(l) for l in open(p, encoding="utf-8-sig") if l.strip()]
pd.set_option("display.max_colwidth", 80)

In [ ]:
RUN = ROOT / "runs/sql_v8_pool"                                             # a train run with trace.jsonl (predict --data data/train)
SUBS = {"v6 (0.63)": "runs/sub_v6", "v8 (0.639)": "runs/sub_v8", "v9 (0.683)": "runs/sub_v9"}   # public submissions and their scores
from datamate import scoring, voting
from datamate.hints import DEBT_WORDS
from datamate.ingest import norm
gold = {a["id"]: a for a in read(DATA / "train/answers.jsonl")}
trace = {r["id"]: r for r in read(RUN / "trace.jsonl")}
print(len(trace), "traced questions")

## 1. Điểm của lượt chạy train

`scoring.evaluate` cho PA, độ chính xác khi trả lời, khoảng tin cậy bootstrap theo shop và phân rã theo loại đáp án.

In [ ]:
questions = {q["id"]: q for q in read(DATA / "train/questions.jsonl")}
pred = {i: r["final"] for i, r in trace.items()}
report, rows = scoring.evaluate({i: gold[i] for i in trace}, pred, questions)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in report.items() if k in ("N", "correct", "wrong", "abstain", "PA", "precision", "coverage")})
print("PA 95% CI (shop bootstrap):", [round(x, 3) for x in report["PA_cluster_bootstrap_95"]])
pd.DataFrame(report["breakdowns"]["gold_type"]).T[["N", "correct", "wrong", "abstain", "PA"]].sort_values("N", ascending=False)

In [ ]:
by_shop = pd.DataFrame(report["breakdowns"]["business_id"]).T[["N", "correct", "wrong", "abstain", "PA"]].sort_values("PA")
by_shop.plot.barh(y="PA", figsize=(6, 7), legend=False, title="PA by shop"); by_shop.head(8)

## Nhóm lỗi

Mỗi câu chưa đúng thuộc một nhóm: không mẫu nào đúng (cần model/prompt tốt hơn), có mẫu đúng nhưng nhóm khác thắng phiếu, hoặc nhóm đúng
thắng nhưng dưới ngưỡng nên bỏ trống (cần bỏ phiếu/ngưỡng tốt hơn).

In [ ]:
def kind(r):
    g = gold[r["id"]]
    s = scoring.score(r["final"], g)
    if s == 1: return "right"
    if not any(p and scoring.score(p, g) == 1 for p in (x["pred"] for x in r["samples"])): return "no right sample"
    if s == -1: return "wrong: a wrong group won"
    top_right = bool(r.get("vote")) and scoring.score(r["vote"], g) == 1
    return "abstain: right group on top, below threshold" if top_right else "abstain: right sample, another group on top"
kinds = Counter(kind(r) for r in trace.values())
pd.Series(kinds).sort_values(ascending=False).to_frame("questions")

## Ngưỡng bỏ phiếu và oracle@K

PA khi đổi ngưỡng đồng thuận (bỏ phiếu lại trên các mẫu đã lưu), và tỷ lệ câu có ít nhất một mẫu đúng (trần của việc chọn đáp án).

In [ ]:
def replay(r, tau):
    texts, preds = [s["text"] for s in r["samples"]], [s["pred"] for s in r["samples"]]
    pool = voting.debt_pool(texts, 8, bool(DEBT_WORDS.search(norm(r["question"]))))
    ans, conf = voting.vote([preds[j] for j in pool], len(pool))
    return voting.final_answer(ans, conf, tau, r["final"].get("reason") == "period outside the shop's data dates", True)
taus = [0.25, 0.375, 0.5, 0.625, 0.75]
sweep = pd.Series({t: sum(scoring.score(replay(r, t), gold[i]) for i, r in trace.items()) / len(trace) for t in taus}, name="PA")
oracle = sum(any(p and scoring.score(p, gold[i]) == 1 for p in (s["pred"] for s in r["samples"])) for i, r in trace.items()) / len(trace)
print(f"oracle (any sample right): {oracle:.3f}"); sweep.to_frame()

## Theo dạng câu hỏi

In [ ]:
from datamate import hints as H
shape = lambda q: ("debt" if DEBT_WORDS.search(norm(q)) else "profit" if H.PROFIT.search(norm(q)) else "trips" if H.TRIPS.search(norm(q))
                   else "lots" if H.LOT.search(norm(q)) else "other")
df = pd.DataFrame([{"shape": shape(r["question"]), "score": scoring.score(r["final"], gold[i])} for i, r in trace.items()])
df.groupby("shape")["score"].agg(n="size", PA="mean").sort_values("n", ascending=False).round(3)

## 2. So các bản nộp public (không nhãn)

Điểm public chỉ có ở mức tổng; ở đây chỉ so các file với nhau: số câu trả lời, mức trùng đáp án, và chỗ khác nhau theo shop.

In [ ]:
def key(a):
    t, x = a.get("type"), a.get("answer")
    if t == "abstain" or (x is None and t != "unanswerable"): return "ABSTAIN"
    return "UNANSWERABLE" if t == "unanswerable" else json.dumps([t, round(x, 2) if isinstance(x, float) else x], ensure_ascii=False, sort_keys=True)
pq = {q["id"]: q for q in read(DATA / "public_test/questions.jsonl")}
subs = {name: {r["id"]: key(r) for r in read(ROOT / p / "submission.jsonl")} for name, p in SUBS.items() if (ROOT / p / "submission.jsonl").exists()}
pd.DataFrame({n: {"answered": sum(v != "ABSTAIN" for v in s.values()), "unanswerable": sum(v == "UNANSWERABLE" for v in s.values())} for n, s in subs.items()}).T

In [ ]:
names = list(subs)
pd.DataFrame([[sum(subs[a][i] == subs[b][i] for i in pq) / len(pq) for b in names] for a in names], index=names, columns=names).style.format("{:.1%}")

In [ ]:
a, b = names[-2], names[-1]
pd.DataFrame([{"shop": q["business_id"], "changed": subs[a][i] != subs[b][i], "newly answered": subs[a][i] == "ABSTAIN" != subs[b][i],
               "newly abstained": subs[b][i] == "ABSTAIN" != subs[a][i]} for i, q in pq.items()]).groupby("shop").sum()